# Measurements preparations

## Preamble

In [24]:
import pandas as pd

from datetime import datetime
import ipaddress

## IPv4

In [30]:
ts = datetime(2025, 8, 27)

In [31]:
anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")

display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i..."
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '..."
2,1.1.8.0/24,1,1,3,1,1,False,1.1.8.0/24,149511_138421,NaN
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id..."


35833


In [32]:
ipv4_pdf = pd.read_csv(f"internet_address_verfploeter_hitlist_it113w-{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
ipv4_pdf.columns = ["ipv4"]

display(ipv4_pdf.head(5))
print(len(ipv4_pdf))

,ipv4
0,1.0.0.0
1,1.0.4.1
2,1.0.5.1
3,1.0.6.1
4,1.0.7.1


5788894


In [33]:
ipv4_pdf["prefix"] = ipv4_pdf["ipv4"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")

display(ipv4_pdf.head(5))

,ipv4,prefix
0,1.0.0.0,1.0.0.0/24
1,1.0.4.1,1.0.4.0/24
2,1.0.5.1,1.0.5.0/24
3,1.0.6.1,1.0.6.0/24
4,1.0.7.1,1.0.7.0/24


In [34]:
responsive_ipv4_pdf = anycast_ipv4_pdf.merge(ipv4_pdf, on="prefix", how="inner")

display(responsive_ipv4_pdf.head(5))
print(len(responsive_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations,ipv4
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i...",1.0.0.0
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '...",1.1.1.0
2,1.1.8.0/24,1,1,3,1,1,False,1.1.8.0/24,149511_138421,NaN,1.1.8.8
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'...",1.10.10.10
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id...",1.12.0.0


34111


In [35]:
responsive_ipv4_pdf["ipv4"].to_csv("responsive_anycast_ipv4_addresses.csv", index=False, header=False)

## IPv6

In [37]:
ts_v6 = datetime(2025, 8, 18)

In [38]:
anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '..."
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
2,2001:1218:600d::/48,2,2,0,1,1,2001:1218:600d::/48,278,NaN
3,2001:1218:6035::/48,2,2,0,1,1,2001:1218:6035::/48,278,NaN
4,2001:1218:c01::/48,2,2,2,1,1,2001:1218::/32,278,NaN


7000


In [39]:
ipv6_pdf = pd.read_csv(f"v6_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)

ipv6_pdf.columns = ["ipv6"]
display(ipv6_pdf.head(5))
print(len(ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


6234585


In [40]:
def is_v6_in_prefix(ipv6, prefix):
    try:
        ipv6_addr = ipaddress.IPv6Address(ipv6)
        ipv6_net = ipaddress.IPv6Network(prefix)
        return ipv6_addr in ipv6_net
    except ValueError:
        return False


def ipv6_to_prefix48(ip):
    try:
        # Create a /48 network that contains this IP, non-strict so host bits allowed
        net = ipaddress.IPv6Network(f"{ip}/48", strict=False)
        # Return in same format as your prefix column, e.g. "2001:1201:10::/48"
        return net.with_prefixlen
    except ValueError:
        return None


ipv6_pdf["prefix"] = ipv6_pdf["ipv6"].apply(lambda x: ipv6_to_prefix48(x))
display(ipv6_pdf.head(5))

,ipv6,prefix
0,2620:10a:80ac::45,2620:10a:80ac::/48
1,2001:678:2c:0:194:0:28:7,2001:678:2c::/48
2,2001:678:78:42:ad::53,2001:678:78::/48
3,2001:dce:2000:2::130,2001:dce:2000::/48
4,2001:dce:7000:2::130,2001:dce:7000::/48


In [41]:
responsive_ipv6_pdf = anycast_ipv6_pdf.merge(ipv6_pdf, on="prefix", how="inner")

display(responsive_ipv6_pdf.head(5))
print(len(responsive_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations,ipv6
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '...",2001:1201:10::1
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1201::1
2,2001:1218:600d::/48,2,2,0,1,1,2001:1218:600d::/48,278,NaN,2001:1218:600d::1
3,2001:1218:6035::/48,2,2,0,1,1,2001:1218:6035::/48,278,NaN,2001:1218:6035::1
4,2001:1218:c01::/48,2,2,2,1,1,2001:1218::/32,278,NaN,2001:1218:c01:203:237::250


6904


In [42]:
responsive_ipv6_pdf["ipv6"].to_csv("responsive_anycast_ipv6_addresses.csv", index=False, header=False)